In [1]:
import pandas as pd

In [3]:
df = pd.read_csv(r'C:\Users\Ganesh\OneDrive\Desktop\dataset\soler_pred\Solar_Energy_Production.csv')
df.head()

,name,id,address,date,kWh,public_url,installationDate,uid
0,Calgary Fire Hall Headquarters,314106,1212 42 AV SE,2017/09/11 08:00:00 AM,1.130,https://monitoring.solaredge.com/solaredge-web...,2016/11/07,3141062017-09-11 08:00:00
1,Calgary Fire Hall Headquarters,314106,1212 42 AV SE,2017/09/11 09:00:00 AM,2.340,https://monitoring.solaredge.com/solaredge-web...,2016/11/07,3141062017-09-11 09:00:00
2,Calgary Fire Hall Headquarters,314106,1212 42 AV SE,2017/09/11 10:00:00 AM,3.656,https://monitoring.solaredge.com/solaredge-web...,2016/11/07,3141062017-09-11 10:00:00
3,Calgary Fire Hall Headquarters,314106,1212 42 AV SE,2017/09/11 11:00:00 AM,4.577,https://monitoring.solaredge.com/solaredge-web...,2016/11/07,3141062017-09-11 11:00:00
4,Calgary Fire Hall Headquarters,314106,1212 42 AV SE,2017/09/11 12:00:00 PM,6.506,https://monitoring.solaredge.com/solaredge-web...,2016/11/07,3141062017-09-11 12:00:00


In [4]:
import numpy as np
import pandas as pd


def load_and_clean_solar_data(file_path):
    """Local Desktop path se raw CSV load, clean, aur feature eng. karta hai."""
    print(f"Loading data from: {file_path}")

    # Load Raw CSV
    df = pd.read_csv(file_path)

    # Automatically detect timestamp column (case-insensitive search)
    date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
    if date_cols:
        time_col = date_cols[0]
        df[time_col] = pd.to_datetime(df[time_col])
        df.set_index(time_col, inplace=True)
    else:
        # Fallback if first column is timestamp
        df.index = pd.to_datetime(df.iloc[:, 0])

    df = df.sort_index()

    # Automatically detect target power column
    target_cols = [
        c
        for c in df.columns
        if "solar" in c.lower()
        or "power" in c.lower()
        or "energy" in c.lower()
        or "kwh" in c.lower()
        or "mw" in c.lower()
    ]
    target_col = target_cols[0] if target_cols else df.columns[0]
    print(f"Target Column Identified: '{target_col}'")

    # Clean duplicates & handle missing values via Time Interpolation
    df = df[~df.index.duplicated(keep="first")]
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df[target_col] = df[target_col].interpolate(method="time").clip(lower=0)

    # Feature Engineering (Cyclical Encodings & Lag Features)
    df["hour"] = df.index.hour
    df["dayofweek"] = df.index.dayofweek
    df["month"] = df.index.month

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

    # Lag & Rolling Features
    df["lag_1h"] = df[target_col].shift(1)
    df["lag_2h"] = df[target_col].shift(2)
    df["lag_24h"] = df[target_col].shift(24)
    df["rolling_mean_6h"] = df[target_col].shift(1).rolling(window=6).mean()

    df.dropna(inplace=True)
    return df, target_col


if __name__ == "__main__":
    # Local Windows Path
    file_path = r"C:\Users\Ganesh\OneDrive\Desktop\dataset\soler_pred\Solar_Energy_Production.csv"
    cleaned_df, target_col = load_and_clean_solar_data(file_path)
    print("Cleaned Data Preview:")
    print(cleaned_df[[target_col, "hour_sin", "lag_1h", "rolling_mean_6h"]].head())

Loading data from: C:\Users\Ganesh\OneDrive\Desktop\dataset\soler_pred\Solar_Energy_Production.csv
Target Column Identified: 'kWh'
Cleaned Data Preview:
                        kWh  hour_sin  lag_1h  rolling_mean_6h
date                                                          
2015-09-02 17:30:00  79.609 -0.965926  87.026        45.275000
2015-09-02 18:30:00  60.377 -1.000000  79.609        52.446667
2015-09-02 19:30:00  64.576 -0.965926  60.377        57.764333
2015-09-02 20:30:00  18.230 -0.866025  64.576        62.955333
2015-09-02 21:30:00  35.959 -0.707107  18.230        61.659000


In [8]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# 1. Dataset Path
DATA_PATH = r"C:\Users\Ganesh\OneDrive\Desktop\dataset\soler_pred\Solar_Energy_Production.csv"

# 2. Data Loading & Preprocessing Function
def load_and_preprocess_solar_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    print(f"--> Reading dataset from: {file_path}")
    df = pd.read_csv(file_path)

    # Detect Date Column
    date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
    if date_cols:
        time_col = date_cols[0]
        df[time_col] = pd.to_datetime(df[time_col])
        df.set_index(time_col, inplace=True)
    else:
        df.index = pd.to_datetime(df.iloc[:, 0])

    df = df.sort_index()

    # Detect Target Column
    target_cols = [c for c in df.columns if any(k in c.lower() for k in ["solar", "power", "energy", "kwh", "mw"])]
    target_col = target_cols[0] if target_cols else df.columns[0]
    print(f"--> Target Column: '{target_col}'")

    # Cleaning
    df = df[~df.index.duplicated(keep="first")]
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df[target_col] = df[target_col].interpolate(method="time").clip(lower=0)

    # Features
    df["hour"] = df.index.hour
    df["dayofweek"] = df.index.dayofweek
    df["month"] = df.index.month
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

    df["lag_1h"] = df[target_col].shift(1)
    df["lag_2h"] = df[target_col].shift(2)
    df["lag_24h"] = df[target_col].shift(24)
    df["rolling_mean_6h"] = df[target_col].shift(1).rolling(window=6).mean()

    df.dropna(inplace=True)
    return df, target_col

# 3. Process Data
df, target_col = load_and_preprocess_solar_data(DATA_PATH)

features = [
    "hour_sin", "hour_cos", "month", "dayofweek",
    "lag_1h", "lag_2h", "lag_24h", "rolling_mean_6h"
]

X = df[features]
y = df[target_col]

# 4. Train-Test Split (80-20)
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"--> Training Model on {len(X_train)} samples...")

# 5. Train XGBoost
model = XGBRegressor(n_estimators=200, learning_rate=0.03, max_depth=6, random_state=42)
model.fit(X_train, y_train)

# 6. Evaluate
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n[EVALUATION METRICS]\n - MAE: {mae:.2f}\n - RMSE: {rmse:.2f}")

# 7. Save Model
os.makedirs("models", exist_ok=True)
artifacts = {
    "model": model,
    "features": features,
    "last_observed_value": float(y.iloc[-1]),
    "target_col_name": target_col,
}

joblib.dump(artifacts, "models/solar_xgboost_artifacts.pkl")
print("\n--> Model Artifacts saved to 'models/solar_xgboost_artifacts.pkl' successfully!")

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# 1. Dataset Path
DATA_PATH = r"C:\Users\Ganesh\OneDrive\Desktop\dataset\soler_pred\Solar_Energy_Production.csv"

# 2. Data Loading & Preprocessing Function
def load_and_preprocess_solar_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    print(f"--> Reading dataset from: {file_path}")
    df = pd.read_csv(file_path)

    # Detect Date Column
    date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
    if date_cols:
        time_col = date_cols[0]
        df[time_col] = pd.to_datetime(df[time_col])
        df.set_index(time_col, inplace=True)
    else:
        df.index = pd.to_datetime(df.iloc[:, 0])

    df = df.sort_index()

    # Detect Target Column
    target_cols = [c for c in df.columns if any(k in c.lower() for k in ["solar", "power", "energy", "kwh", "mw"])]
    target_col = target_cols[0] if target_cols else df.columns[0]
    print(f"--> Target Column: '{target_col}'")

    # Cleaning
    df = df[~df.index.duplicated(keep="first")]
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df[target_col] = df[target_col].interpolate(method="time").clip(lower=0)

    # Features
    df["hour"] = df.index.hour
    df["dayofweek"] = df.index.dayofweek
    df["month"] = df.index.month
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

    df["lag_1h"] = df[target_col].shift(1)
    df["lag_2h"] = df[target_col].shift(2)
    df["lag_24h"] = df[target_col].shift(24)
    df["rolling_mean_6h"] = df[target_col].shift(1).rolling(window=6).mean()

    df.dropna(inplace=True)
    return df, target_col

# 3. Process Data
df, target_col = load_and_preprocess_solar_data(DATA_PATH)

features = [
    "hour_sin", "hour_cos", "month", "dayofweek",
    "lag_1h", "lag_2h", "lag_24h", "rolling_mean_6h"
]

X = df[features]
y = df[target_col]

# 4. Train-Test Split (80-20)
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"--> Training Model on {len(X_train)} samples...")

# 5. Train XGBoost
model = XGBRegressor(n_estimators=200, learning_rate=0.03, max_depth=6, random_state=42)
model.fit(X_train, y_train)

# 6. Evaluate
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n[EVALUATION METRICS]\n - MAE: {mae:.2f}\n - RMSE: {rmse:.2f}")

# 7. Save Model
os.makedirs("models", exist_ok=True)
artifacts = {
    "model": model,
    "features": features,
    "last_observed_value": float(y.iloc[-1]),
    "target_col_name": target_col,
}

joblib.dump(artifacts, "models/solar_xgboost_artifacts.pkl")
print("\n--> Model Artifacts saved to 'models/solar_xgboost_artifacts.pkl' successfully!")

--> Reading dataset from: C:\Users\Ganesh\OneDrive\Desktop\dataset\soler_pred\Solar_Energy_Production.csv
--> Target Column: 'kWh'
--> Training Model on 32575 samples...

[EVALUATION METRICS]
 - MAE: 2787.53
 - RMSE: 18672.29

--> Model Artifacts saved to 'models/solar_xgboost_artifacts.pkl' successfully!
--> Reading dataset from: C:\Users\Ganesh\OneDrive\Desktop\dataset\soler_pred\Solar_Energy_Production.csv
--> Target Column: 'kWh'
--> Training Model on 32575 samples...

[EVALUATION METRICS]
 - MAE: 2787.53
 - RMSE: 18672.29

--> Model Artifacts saved to 'models/solar_xgboost_artifacts.pkl' successfully!
